# Laboratorio 1 de Modelos y Simulación.
## Modelado de trayectorias más complejas

**Objetivos de este laboratorio:**

* diseñar y programar trayectorias de simulación más complejas en SimPy
* construir un modelo completo de simulación por computadora.
* usar funciones de Python para organizar tu experimentación y escenarios.


Este Laboratorio solicita que programen la lógica de una Unidad de Lesiones Menores (MIU). El notebook incluye varias clases y código para ayudarles con la programación del modelo.  

* Clases de distribución, como la distribución `Exponential`.
* Una clase `Scenario` para permitirte pasar parámetros al modelo.
* Clases plantilla para representar la Unidad de Lesiones Menores y el proceso que siguen los pacientes.

-----

# Importaciones

In [3]:
!pip install simpy

import simpy
simpy.__version__

'4.1.1'

In [2]:
import numpy as np
import pandas as pd
import itertools
import math
import matplotlib.pyplot as plt

# Descripción del problema: modelado de una unidad de lesiones menores

Los ejercicios de este laboratorio se centran en un modelo de una sala de urgencias (SU), las cuales ofrecen evaluación y tratamiento para lesiones no potencialmente mortales; por ejemplo, cortes y fracturas.

El siguiente diagrama de proceso muestra un mapa de alto nivel de una SU.  
* Los pacientes llegan según un proceso de Poisson y esperan el triaje de enfermería en un área de triaje.
* Una vez completado el triaje, el paciente espera un cubículo de valoración y tratamiento (hay **seis**), donde es atendido por una enfermera (aquí suponemos que la enfermera no es el cuello de botella y no modelamos su movimiento ni su disponibilidad).  
* Aproximadamente el **45%** de los pacientes requiere pruebas diagnósticas (por ejemplo, una radiografía u otra imagen) además de la valoración.  
* Aproximadamente el **75%** de las pruebas diagnósticas se solicita por la enfermera al inicio de la consulta; el resto comienza su espera después de la valoración inicial.
* Los pacientes que se someten a diagnósticos son atendidos por una enfermera por segunda vez tras finalizar sus pruebas.

Las duraciones de las actividades son:

| Actividad | Nombre de la actividad         | Distribución | Media (min) | Desv. estándar (minutos) |
|----------|----------------------|--------------|-------------|------------------------|
| 1        | IAT                   | Exponential   | 5.0          |                        |
| 2        | Primera valoración    | Lognormal     | 10           | 3                      |
| 3        | Tiempo de espera para diagnóstico | Exponential | 30           |                        |
| 4        | Segunda valoración    | Lognormal     | 10           | 5                      |

y

| Actividad | Nombre de la actividad | Distribución | Mínimo | Moda | Máximo |
|----------|----------------------|--------------|-------------|------------------------|----- |
| 5        | Triaje               | Triangular   | 2.0           |   5.0                      | 10.0|
| 6        | Diagnósticos         | Triangular   | 10.0          | 15.0                       | 20.0 |





![image](images/SU.png)

# Clases de distribución

Para ayudarte a construir tu modelo, el notebook incluye algunas clases de distribución ya escritas que puedes usar para configurar el muestreo. Eres libre de usarlas, pero puedes optar por no hacerlo si prefieres otro enfoque.

In [4]:
class Exponential:
    '''
    Clase de conveniencia para la distribución exponencial.
    Agrupa los parámetros de la distribución, la semilla y el generador aleatorio.
    '''
    def __init__(self, mean, random_seed=None):
        '''
        Constructor

        Parámetros:
        ------------
        mean: float
            Media de la distribución exponencial.

        random_seed: int, opcional (por defecto=None)
            Semilla aleatoria para reproducir las muestras. Si se deja como None,
            se genera una muestra única.
        '''
        self.rand = np.random.default_rng(seed=random_seed)
        self.mean = mean

    def sample(self, size=None):
        '''
        Genera una muestra de la distribución exponencial.

        Parámetros:
        ------------
        size: int, opcional (por defecto=None)
            Número de muestras a devolver. Si size=None, se devuelve una única muestra.
        '''
        return self.rand.exponential(self.mean, size=size)

class Triangular:
    '''
    Clase de conveniencia para la distribución triangular.
    Agrupa los parámetros de la distribución, la semilla y el generador aleatorio.
    '''
    def __init__(self, low, mode, high, random_seed=None):
        self.rand = np.random.default_rng(seed=random_seed)
        self.low = low
        self.high = high
        self.mode = mode

    def sample(self, size=None):
        return self.rand.triangular(self.low, self.mode, self.high, size=size)

class Bernoulli:
    '''
    Clase de conveniencia para la distribución de Bernoulli.
    Agrupa los parámetros de la distribución, la semilla y el generador aleatorio.
    '''
    def __init__(self, p, random_seed=None):
        '''
        Constructor

        Parámetros:
        ------------
        p: float
            Probabilidad de obtener un 1.

        random_seed: int, opcional (por defecto=None)
            Semilla aleatoria para reproducir las muestras. Si se deja como None,
            se genera una muestra única.
        '''
        self.rand = np.random.default_rng(seed=random_seed)
        self.p = p

    def sample(self, size=None):
        '''
        Genera una muestra de la distribución de Bernoulli.

        Parámetros:
        ------------
        size: int, opcional (por defecto=None)
            Número de muestras a devolver. Si size=None, se devuelve una única muestra.
        '''
        return self.rand.binomial(n=1, p=self.p, size=size)

class Lognormal:
    """
    Encapsula una distribución lognormal.
    """
    def __init__(self, mean, stdev, random_seed=None):
        """
        Parámetros:
        ------------
        mean = media de la distribución lognormal.
        stdev = desviación estándar de la distribución lognormal.
        """
        self.rand = np.random.default_rng(seed=random_seed)
        mu, sigma = self.normal_moments_from_lognormal(mean, stdev**2)
        self.mu = mu
        self.sigma = sigma

    def normal_moments_from_lognormal(self, m, v):
        '''
        Devuelve mu y sigma de la distribución normal subyacente
        a una lognormal con media m y varianza v.
        Fuente: https://blogs.sas.com/content/iml/2014/06/04/simulate-lognormal
        -data-with-specified-mean-and-variance.html

        Parámetros:
        ------------
        m = media de la distribución lognormal.
        v = varianza de la distribución lognormal.

        Retorna:
        ------------
        (float, float)
        '''
        phi = math.sqrt(v + m**2)
        mu = math.log(m**2/phi)
        sigma = math.sqrt(math.log(phi**2/m**2))
        return mu, sigma

    def sample(self):
        """
        Genera una muestra de la distribución lognormal.
        """
        return self.rand.lognormal(self.mu, self.sigma)

In [5]:
# ejemplo de código para usar las distribuciones

# cada distribución tiene un constructor único al que puedes pasar una semilla para controlar el muestreo
triage_dist = Triangular(2, 5, 10, random_seed=42)

# cada distribución tiene una firma de método `sample` idéntica
delay = triage_dist.sample()
print(delay)

6.993048377881435


# Utilidades

En la simulación se usa la función `trace` en lugar de `print`.  De este modo puedes controlar si la simulación genera una trazabilidad de eventos estableciendo `TRACE = True` o desactivar la trazabilidad de eventos con `TRACE = False`.

In [6]:
def trace(msg):
    '''
    Utility function for printing simulation
    set the TRACE constant to FALSE to
    turn tracing off.

    Params:
    -------
    msg: str
        string to print to screen.
    '''
    if TRACE:
        print(msg)

def trace(msg):
    '''
    Función para imprimir el seguimiento de la simulación.
    Establece la constante TRACE en FALSE para
    desactivar el seguimiento.

    Parámetros:
    ------------
    msg: str
        cadena de texto que se imprime en pantalla.
    '''
    if TRACE:
        print(msg)


# Parámetros del modelo y la clase `Scenario`

* Las constantes de abajo proporcionan datos codificados que representan el caso base de la sala de urgencias.   
* En la celda situada debajo de los parámetros encontrarás una clase `Scenario`.  Esta utiliza los parámetros por defecto para configurar el escenario base.  Recuerda que es buena práctica pasar todos tus parámetros a tu modelo de simulación en un **contenedor**.  Una clase es una manera flexible de hacerlo.

In [7]:
# Estos son los parámetros para una ejecución del modelo en el caso base.
# Nota: si cambias estos parámetros, tu modelo ejecutará un nuevo 'escenario'.

# conteo de recursos
N_CUBICLES = 6
N_BAYS = 1

# tiempo entre llegadas en minutos (exponencial)
MEAN_IAT = 5

# triaje (triangular)
TRIAGE_LOW = 2
TRIAGE_MODE = 5
TRIAGE_HIGH = 10

# valoración (lognormal)
ASSESS_MEAN = 10
ASSESS_STD = 3

# segunda valoración (lognormal)
ASSESS_MEAN_2 = 15
ASSESS_STD_2 = 5

# diagnósticos (bernoulli)
PROB_DIAG = 0.45

# tiempo de espera para diagnóstico (exponencial)
DIAG_WT_MEAN = 20

# tiempo de proceso de diagnóstico (triangular)
DIAG_PT_LOW = 10
DIAG_PT_MODE = 15
DIAG_PT_HIGH = 20

# porcentaje en paralelo (bernoulli)
P_PARALLEL = 0.75

# SEMILLAS para reproducir los resultados de una ejecución
REPRODUCIBLE_RUN = True

if REPRODUCIBLE_RUN:
    SEEDS = [42, 101, 1066, 1966, 2013, 999, 1444, 2016]
else:
    SEEDS = [None, None, None, None, None, None, None, None]


In [8]:
class Scenario:
    '''
    Clase contenedora de parámetros para el modelo de unidad de lesiones menores.
    '''
    def __init__(self, name=None):
        '''
        El método init configura los valores predeterminados.

        Parámetros:
        ------------
        name - str o None
            nombre opcional para el escenario
        '''
        # nombre opcional
        self.name = name

        # puestos de triaje
        self.triage_bays = N_BAYS

        # cubículos de valoración y tratamiento
        self.minor_cubicles = N_CUBICLES

        # distribución de tiempo entre llegadas
        self.arrival_dist = Exponential(MEAN_IAT, random_seed=SEEDS[0])

        # distribución de triaje
        self.triage_dist = Triangular(TRIAGE_LOW, TRIAGE_MODE, TRIAGE_HIGH,
                                      random_seed=SEEDS[1])

        # distribución de valoración
        self.assessment_dist = Lognormal(ASSESS_MEAN, ASSESS_STD,
                                         random_seed=SEEDS[2])

        # diagnósticos: probabilidad de que el paciente requiera imágenes, etc.
        self.n_diagnostics_minor = Bernoulli(PROB_DIAG, random_seed=SEEDS[3])

        # distribuciones de tiempo de espera y proceso para diagnósticos
        self.diag_wait_dist = Exponential(DIAG_WT_MEAN, random_seed=SEEDS[4])
        self.diag_dist = Triangular(DIAG_PT_LOW, DIAG_PT_MODE, DIAG_PT_HIGH,
                                    random_seed=SEEDS[5])

        # probabilidad de que los diagnósticos se realicen en paralelo con la valoración de enfermería
        self.p_diag_parallel = Bernoulli(P_PARALLEL, random_seed=SEEDS[6])

        # distribución de la segunda valoración
        self.assessment_dist_2 = Lognormal(ASSESS_MEAN_2, ASSESS_STD_2,
                                           random_seed=SEEDS[7])


# Ejercicio 1: Construcción del modelo

**Tarea:**
* Programa la lógica del proceso que sigue un paciente en la unidad de lesiones menores.
* Ejecuta el modelo usando el script proporcionado.
* Modifica el script que ejecuta el modelo para que calcule las siguientes medidas de desempeño:
  * Tiempo en el sistema (desde la llegada hasta la salida)
  * Tiempo hasta enfermería (desde la llegada hasta el inicio de la primera valoración)
  * Tiempo hasta triaje (desde la llegada hasta el inicio del triaje)
  * Objetivo de cuatro horas (si ese paciente pasó menos de 4 horas en la SU).
  
**Guias generales:**
* Se te ha proporcionado código esqueletal y clases para ayudarte a completar esta tarea.  
* Recuerda volver al mapa de proceso al inicio del notebook para refrescar la lógica.

**Modelado del proceso de triaje:**
* Lo primero que debes programar es el triaje.  El paciente necesita solicitar un recurso, esperar a que esté disponible y luego muestrear una duración de triaje de la distribución correcta.  
* Recuerda que para solicitar un recurso debes seguir un patrón de SimPy como este:

```python
with self.triage_bays.request() as req:
    yield req
```

* Recuerda que SimPy funciona utilizando **generadores de Python**. Usa la palabra clave **yield** junto con el método `simpy.Environment.timeout()` para crear demoras en el proceso. Por ejemplo,

```python
triage_delay = self.triage_dist.sample()
yield self.env.timeout(triage_delay)
```

**Modelado de la primera valoración y de los procesos de diagnóstico**:
* La parte complicada de este ejercicio es modelar el porcentaje de pruebas diagnósticas que ocurren en paralelo a la valoración de enfermería.  Algunos consejos:
  * Para este ejercicio, debes suponer que el tiempo de la primera valoración comienza en el momento en que se asigna un cubículo al paciente.
  * Recuerda que solo el 45% de los pacientes necesita diagnósticos y que la enfermera ordena que el 75% de las pruebas diagnósticas se ejecuten en paralelo a la primera valoración.
  * Manténlo simple y construye el modelo de forma incremental; es decir, crea primero un modelo sencillo que funcione y luego añade más detalle.  
  * Una primera tarea podría ser modelar el proceso en el que la valoración siempre va seguida de diagnósticos.  Luego puedes limitarlo a un porcentaje de pacientes.
  * Cuando los diagnósticos van en paralelo, solo necesitas modelar un retraso que sea el máximo entre el tiempo de valoración o la suma de tiempo de espera de diagnóstico + duración del diagnóstico.
* Un paciente mantiene su cubículo mientras se somete a diagnósticos.  No permitas que el cubículo se asigne a un nuevo paciente.

**Seguimiento del tiempo en el modelo**
* Necesitarás llevar registro de cuánto tardan los pacientes en completar procesos en el modelo, p. ej., tiempo hasta triaje y tiempo hasta la primera valoración de enfermería.
* Un ejemplo de cómo registrar un tiempo en el modelo es el siguiente:

```python
arrival_time = self.env.now
```

* Cuando necesites calcular un tiempo hasta algo solo debes tomar la diferencia entre el tiempo actual y el tiempo de llegada. Por ejemplo:

```python
self.time_to_triage = self.env.now - arrival_time
```

**Cálculo de medidas de desempeño al final de la ejecución**:
* En este ejercicio es más sencillo calcular tus medidas de desempeño al **final** de la ejecución del modelo.
* Deben mantiener una lista de objetos. Itera (haz un loop) sobre la lista y accede a la medida de desempeño relevante de cada paciente. Luego puedes calcular la metrica solicitada.

In [17]:
    def assessment(self):
        '''
        Simula el proceso para SU.

        1. Solicitar y esperar triaje.
        2. Solicitar y esperar un cubículo menor.
        3. Valoración inicial.
        4. Diagnósticos (en paralelo o después del paso 3).
        5. Segunda valoración si el paciente se sometió a diagnósticos.
        6. Salida del sistema.
        '''
        # --------------------- Tiempo de llegada ---------------------
        arrival_time = self.env.now

        # --------------------- 1. Triaje ----------------------------
        with self.triage_bays.request() as req:
            yield req  # Esperar turno de triaje

            # Tiempo esperado hasta inicio del triaje
            self.time_to_triage = self.env.now - arrival_time

            # Duración del triaje
            triage_time = float(self.triage_dist.sample())
            yield self.env.timeout(triage_time)

        # --------------------- 2. Cubículo --------------------------
        with self.minor_cubicles.request() as req2:
            yield req2  # Esperar cubículo

            # Tiempo hasta enfermería = tiempo desde llegada hasta conseguir cubículo
            self.time_to_nurse = self.env.now - arrival_time

            # --------------------- 3. Primera valoración -------------
            assess_time = float(self.assessment_dist.sample())

            # ¿Este paciente requiere diagnóstico? 45% probabilidad
            need_diag = int(self.n_diagnostics_dist.sample())

            if need_diag == 1:
                # ---------------- Diagnóstico requerido ----------------
                parallel = int(self.p_diag_parallel.sample())  # ¿en paralelo? 75%

                if parallel == 1:
                    # Diagnóstico EN PARALELO con la primera valoración
                    diag_wait = float(self.diag_wait_dist.sample())
                    diag_proc = float(self.diag_dist.sample())
                    total_diag = diag_wait + diag_proc

                    # Procesarlos en paralelo ⇒ demora el máximo de ambos
                    yield self.env.timeout(max(assess_time, total_diag))

                else:
                    # Diagnóstico DESPUÉS de primera valoración
                    yield self.env.timeout(assess_time)

                    diag_wait = float(self.diag_wait_dist.sample())
                    diag_proc = float(self.diag_dist.sample())
                    yield self.env.timeout(diag_wait + diag_proc)

                # Segunda valoración después de diagnóstico
                assess2_time = float(self.assessment_dist_2.sample())
                yield self.env.timeout(assess2_time)

            else:
                # ---------------- No requiere diagnóstico ---------------
                yield self.env.timeout(assess_time)

        # --------------------- 6. Salida ---------------------------
        self.time_in_system = self.env.now - arrival_time
        self.four_hour_target = 1 if self.time_in_system <= 240 else 0


In [18]:
class Patient:
    '''
    Simula el proceso para un paciente con una lesión menor.
    '''
    def __init__(self, identifier, env, args):
        '''
        Método constructor.
        '''
        self.identifier = identifier
        self.env = env

        # parámetros de triaje
        self.triage_bays = args.triage_bays
        self.triage_dist = args.triage_dist

        # parámetros de valoración menor
        self.minor_cubicles = args.minor_cubicles
        self.assessment_dist = args.assessment_dist
        self.assessment_dist_2 = args.assessment_dist_2

        # diagnósticos
        self.n_diagnostics_dist = args.n_diagnostics_minor
        self.p_diag_parallel = args.p_diag_parallel
        self.diag_wait_dist = args.diag_wait_dist
        self.diag_dist = args.diag_dist

        # métricas individuales del paciente
        self.time_to_triage = 0.0
        self.time_to_nurse = 0.0
        self.time_in_system = 0.0
        self.four_hour_target = 0.0

    def assessment(self):
        '''
        Simula el proceso para la unidad de lesiones menores.
        '''
        # Tiempo de llegada
        arrival_time = self.env.now

        # 1. Triaje
        with self.triage_bays.request() as req:
            yield req
            self.time_to_triage = self.env.now - arrival_time
            triage_time = float(self.triage_dist.sample())
            yield self.env.timeout(triage_time)

        # 2. Cubículo de valoración
        with self.minor_cubicles.request() as req2:
            yield req2
            self.time_to_nurse = self.env.now - arrival_time

            # 3. Primera valoración
            assess_time = float(self.assessment_dist.sample())
            need_diag = int(self.n_diagnostics_dist.sample())

            if need_diag == 1:
                parallel = int(self.p_diag_parallel.sample())
                diag_wait = float(self.diag_wait_dist.sample())
                diag_proc = float(self.diag_dist.sample())
                total_diag = diag_wait + diag_proc

                if parallel == 1:
                    yield self.env.timeout(max(assess_time, total_diag))
                else:
                    yield self.env.timeout(assess_time)
                    yield self.env.timeout(total_diag)

                assess2_time = float(self.assessment_dist_2.sample())
                yield self.env.timeout(assess2_time)
            else:
                yield self.env.timeout(assess_time)

        # 4. Salida del sistema
        self.time_in_system = self.env.now - arrival_time
        self.four_hour_target = 1 if self.time_in_system <= 240 else 0

class InjuryUnit:
    '''
    Modelo de una unidad de lesiones menores.
    '''
    def __init__(self, env, args):
        '''
        Constructor.

        Parámetros:
        ------------
        env: simpy.Environment

        args: Scenario
            clase contenedora para las entradas del modelo de simulación.
        '''
        self.env = env
        self.args = args
        self.init_model_resources(args)
        self.patients = []


    def init_model_resources(self, args):
        '''
        Configura los objetos de recursos de simpy.

        Parámetros:
        ------------
        args - Scenario
            Contenedor de parámetros de simulación.
        '''
        args.triage_bays = simpy.Resource(self.env,
                                          capacity=args.triage_bays)
        args.minor_cubicles = simpy.Resource(self.env,
                                             capacity=args.minor_cubicles)


    def arrivals_generator(self):
        '''
        El tiempo entre llegadas (IAT) sigue una distribución exponencial.

        Parámetros:
        ------------
        env: simpy.Environment

        args: Scenario
            Clase contenedora para los datos de entrada del modelo.
        '''
        for patient_count in itertools.count(start=1):
            inter_arrival_time = self.args.arrival_dist.sample()
            yield self.env.timeout(inter_arrival_time)

            trace(f'El paciente {patient_count} llego a las: {self.env.now:.3f}')

            # crear un nuevo paciente con lesión menor y pasarle env y args
            new_patient = Patient(patient_count, self.env, self.args)

            # mantener un registro del paciente para el cálculo de resultados
            self.patients.append(new_patient)

            # iniciar el proceso de la unidad de lesiones menores para este paciente
            self.env.process(new_patient.assessment())


## Script para ejecutar el modelo

In [19]:
# duración de la ejecución en minutos
RUN_LENGTH = 1440

# desactivar el seguimiento
TRACE = False

# crear el entorno de simpy
env = simpy.Environment()

# escenario base con parámetros predeterminados
default_args = Scenario()

# crear el modelo
model = InjuryUnit(env, default_args)

# configurar el proceso
env.process(model.arrivals_generator())

env.run(until=RUN_LENGTH)
print(f'Final de la corrida. tiempo de simulación = {env.now}')

####### Tu código aquí ######################################
# Calcular e imprimir las métricas de desempeño

# ======= Cálculo de métricas =======
times_in_system = [p.time_in_system for p in model.patients]
times_to_triage = [p.time_to_triage for p in model.patients]
times_to_nurse = [p.time_to_nurse for p in model.patients]
four_hour_targets = [p.four_hour_target for p in model.patients]

print("\n Resultados del modelo ")
print(f"Pacientes atendidos: {len(model.patients)}")
print(f"Tiempo promedio en el sistema: {np.mean(times_in_system):.2f} minutos")
print(f"Tiempo promedio hasta triaje: {np.mean(times_to_triage):.2f} minutos")
print(f"Tiempo promedio hasta enfermería: {np.mean(times_to_nurse):.2f} minutos")
print(f"Cumplimiento del objetivo de 4 horas: {np.mean(four_hour_targets) * 100:.2f}%")





Final de la corrida. tiempo de simulación = 1440

 Resultados del modelo 
Pacientes atendidos: 302
Tiempo promedio en el sistema: 144.69 minutos
Tiempo promedio hasta triaje: 118.30 minutos
Tiempo promedio hasta enfermería: 125.99 minutos
Cumplimiento del objetivo de 4 horas: 61.92%


---
# Ejercicio 2: Uso del modelo para la experimentación

¡Ahora que tienes tu modelo puedes usarlo para experimentar!  Vamos a investigar los siguientes escenarios

* 1 puesto de triaje adicional
* 1 cubículo adicional
* 5 cubículos adicionales
* 2 puestos de triaje adicionales + 1 cubículo adicional

## Uso de funciones para organizar la experimentación.

Hasta ahora hemos ejecutado el modelo desde un script.  En la práctica, es buena idea usar funciones que te ayuden a ejecutar escenarios de forma eficiente.  

### `single_model_run()`

La función `single_model_run` se ha proporcionado para ayudarte a hacerlo.  Échale un vistazo primero.  Observa que es muy similar a tu script anterior.   Acepta un parámetro `scenario` que contiene todos los parámetros que deseas usar para la ejecución.  Cuando finaliza la ejecución del modelo devuelve las medidas de desempeño deseadas. Esto separa la creación de los escenarios de tu código de ejecución del modelo.

### `get_scenarios()`

Una manera sensata de organizar la experimentación es crear todos los escenarios por adelantado, almacenarlos en una `list` o `dict` de Python y luego iterar sobre ellos pasando cada uno a la función `single_model_run`.  La función `get_scenarios` ya ha sido codificada para ti e incluye un ejemplo de creación de escenarios.  Como parte del ejercicio deberás completar el código y añadir los escenarios adicionales.

> Nota: en la práctica, si trabajas con modelos grandes y complejos, puede convenirte guardar todos tus escenarios y parámetros en un archivo CSV y luego leerlos.  Aquí tenemos un modelo sencillo, así que codificamos todo explícitamente para mayor claridad.

**Tarea**
* Lee y comprueba que entiendes las funciones y el script proporcionados.
* Completa la función `get_scenarios()` para que tengas resultados para todos los escenarios.
* ¡Ejecuta todos los escenarios!

In [20]:
def single_model_run(scenario, run_length):
    '''
    Realiza una única ejecución del modelo y devuelve los resultados.

    Parámetros:
    ------------

    env = Simpy.Environment

    scenario - objeto Scenario
        El escenario o conjunto de parámetros a ejecutar.

    run_length - int
        La duración de la ejecución de la simulación.


    Retorna:
        Tupla:
        (mean_time_in_system, mean_time_to_nurse, mean_time_to_triage,
         four_hours)
    '''
    env = simpy.Environment()
    # crear el modelo y pasarle el escenario
    model = InjuryUnit(env, scenario)

    # configurar el proceso
    env.process(model.arrivals_generator())

    # ejecutar
    env.run(until=run_length)

    # Métricas de desempeño

    # 1. tiempo medio en el sistema
    mean_time_in_system = np.array([patient.time_in_system
                                    for patient in model.patients]).mean()

    # 2. tiempo medio hasta la primera valoración
    mean_time_to_nurse = np.array([patient.time_to_nurse
                                   for patient in model.patients]).mean()

    # 3. tiempo medio hasta el triaje
    mean_time_to_triage = np.array([patient.time_to_triage
                                   for patient in model.patients]).mean()

    # 4. cumplimiento del objetivo de cuatro horas
    four_hours = np.array([patient.four_hour_target
                           for patient in
                           model.patients]).sum() / len(model.patients)

    # devolver resultados
    return (mean_time_in_system, mean_time_to_nurse, mean_time_to_triage,
            four_hours)

In [21]:
def get_scenarios():
    '''
    Configura los escenarios. Los escenarios a ejecutar se devuelven en un dict{str:Scenario}.
    '''
    scenarios = {}

    # Escenario base
    scenarios['base'] = Scenario()
    scenarios['base'].name = 'base'

    # Escenario 1: +1 puesto de triaje
    scenario_1 = Scenario()
    scenario_1.triage_bays = N_BAYS + 1
    scenario_1.name = 'extra_triage_capacity'
    scenarios['extra_triage_capacity'] = scenario_1

    # Escenario 2: +1 cubículo
    scenario_2 = Scenario()
    scenario_2.minor_cubicles = N_CUBICLES + 1
    scenario_2.name = 'extra_cubicle'
    scenarios['extra_cubicle'] = scenario_2

    # Escenario 3: +5 cubículos
    scenario_3 = Scenario()
    scenario_3.minor_cubicles = N_CUBICLES + 5
    scenario_3.name = 'extra_5_cubicles'
    scenarios['extra_5_cubicles'] = scenario_3

    # Escenario 4: +1 triaje y +1 cubículo
    scenario_4 = Scenario()
    scenario_4.triage_bays = N_BAYS + 1
    scenario_4.minor_cubicles = N_CUBICLES + 1
    scenario_4.name = 'extra_triage_and_cubicle'
    scenarios['extra_triage_and_cubicle'] = scenario_4

    return scenarios


### Script para ejecutar los escenarios.

In [23]:
RUN_LENGTH = 1440  # 1440 minutos = 1 día
TRACE = False

# cargar escenarios
scenarios = get_scenarios()

# recorrer cada escenario y almacenar resultados
results = {}
for name, scenario in scenarios.items():
    print(f'Scenario de simulación: {name}')
    results[name] = single_model_run(scenario, RUN_LENGTH)

print('\nTodas las simulaciones completadas.\n')

# convertir resultados en DataFrame
results = pd.DataFrame(results).T
results.columns = ['Tiempo en el sistema', 'Tiempo hasta enfermería',
                   'Tiempo hasta triaje', 'Desempeño 4 horas (%)']

results


Scenario de simulación: base
Scenario de simulación: extra_triage_capacity
Scenario de simulación: extra_cubicle
Scenario de simulación: extra_5_cubicles
Scenario de simulación: extra_triage_and_cubicle

Todas las simulaciones completadas.



,Tiempo en el sistema,Tiempo hasta enfermería,Tiempo hasta triaje,Desempeño 4 horas (%)
base,144.685081,125.985080,118.297042,0.619205
extra_triage_capacity,99.160479,74.370677,1.834980,0.894040
extra_cubicle,140.905896,122.967824,118.297042,0.632450
extra_5_cubicles,140.170316,122.232244,118.297042,0.635762
extra_triage_and_cubicle,50.148899,22.312434,1.834980,0.973510
